
# Photo-z degeneracy: chi² landscape over redshift and stellar mass

Galaxy photometry is degenerate in redshift and stellar mass — the same
galaxy can look identical at different redshifts if the mass is adjusted.
We mock a star-forming galaxy at z=2.5 with known stellar mass, observe it
in ugrizYJHK bands at S/N=10, then compute χ² on a 2D grid of (z, M*) to
show the classic photo-z degeneracy valley. The figure maps χ² as a heatmap
with 1σ/2σ/3σ contours and overlays the true redshift.

Reference: Bolzonella et al. 2000, A&A, 363, 476 (HYPERZ photometric
redshift); Brammer et al. 2008, ApJ, 686, 1503 (EAZY photometric redshift).


In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # suppress XLA/PjRt C++ INFO+WARNING logs

import warnings

import jax
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.analysis.plotting import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")
warnings.filterwarnings("ignore", message=".*FutureWarning.*")

We use a basic star-forming template: truncated-skew-normal SFH + Calzetti
dust attenuation. Redshift and stellar mass are the free parameters on the grid.



In [ ]:
BANDS = [
    "sdss_u",
    "sdss_g",
    "sdss_r",
    "sdss_i",
    "sdss_z",
    "vista_y",
    "vista_j",
    "vista_h",
    "vista_ks",
]

obs = tengri.Observation(photometry=tengri.Photometry.from_names(BANDS))

# Template: star-forming with age ~2 Gyr, modest dust.
model_template = tengri.SEDModel.build(
    tengri.load_ssp("fsps_prsc_miles_chabrier"),
    observation=obs,
    sfh={
        "type": "tsnorm",
        "*": tengri.FIXED,
        "peak_lbt_gyr": 3.0,
        "width_gyr": 2.0,
        "log_total_mass": 10.0,
        "skew": 0.3,
        "trunc": 10.0,
    },
    dust={
        "type": "two_component",
        "*": tengri.FIXED,
        "tau_diff": 0.3,
        "tau_bc": 0.1,
        "slope": -0.7,
    },
    neb={"type": "cue", "*": tengri.FIXED},
    redshift=tengri.Fixed(0.0),  # will vary on grid
)

In [ ]:
z_true = 2.5
log_mstar_true = 10.5  # ~3.2e10 Msun

key = jax.random.PRNGKey(42)
truth_params = dict(model_template.spec.sample(key))
truth_params.update(
    sfh_tsnorm_peak_lbt_gyr=3.0,
    sfh_tsnorm_width_gyr=2.0,
    sfh_tsnorm_log_total_mass=log_mstar_true,
    sfh_tsnorm_skew=0.3,
    sfh_tsnorm_trunc=10.0,
    dust_tau_diff=0.3,
    dust_tau_bc=0.1,
    dust_slope=-0.7,
    neb_logU=-3.0,
    neb_logZ_gas=0.0,
    redshift=z_true,
)

# Create a model at truth z to generate mock data.
model_truth = tengri.SEDModel.build(
    tengri.load_ssp("fsps_prsc_miles_chabrier"),
    observation=obs,
    sfh={
        "type": "tsnorm",
        "*": tengri.FIXED,
        "peak_lbt_gyr": 3.0,
        "width_gyr": 2.0,
        "log_total_mass": 10.0,
        "skew": 0.3,
        "trunc": 10.0,
    },
    dust={
        "type": "two_component",
        "*": tengri.FIXED,
        "tau_diff": 0.3,
        "tau_bc": 0.1,
        "slope": -0.7,
    },
    neb={"type": "cue", "*": tengri.FIXED},
    redshift=tengri.Fixed(z_true),
)

mock = model_truth.mock(truth_params, snr=10.0, key=key)
flux_obs = np.asarray(mock.flux_obs)
noise_obs = np.asarray(mock.noise)

For each (z, M*) pair, build a model, predict photometry, and compute
χ² = sum((F_obs - F_pred)² / σ²).



In [ ]:
z_grid = np.linspace(0.5, 5.0, 65)
log_mstar_grid = np.linspace(9.0, 11.5, 65)
chi2_grid = np.zeros((z_grid.size, log_mstar_grid.size))

for i, z in enumerate(z_grid):
    for j, log_mstar in enumerate(log_mstar_grid):
        # Build model at this redshift.
        model_z = tengri.SEDModel.build(
            tengri.load_ssp("fsps_prsc_miles_chabrier"),
            observation=obs,
            sfh={
                "type": "tsnorm",
                "*": tengri.FIXED,
                "peak_lbt_gyr": 3.0,
                "width_gyr": 2.0,
                "log_total_mass": 10.0,
                "skew": 0.3,
                "trunc": 10.0,
            },
            dust={
                "type": "two_component",
                "*": tengri.FIXED,
                "tau_diff": 0.3,
                "tau_bc": 0.1,
                "slope": -0.7,
            },
            neb={"type": "cue", "*": tengri.FIXED},
            redshift=tengri.Fixed(z),
        )

        # Predict photometry at this stellar mass.
        grid_params = dict(truth_params)
        grid_params["sfh_tsnorm_log_total_mass"] = log_mstar
        flux_pred = np.asarray(model_z.predict_photometry(grid_params))

        # Compute χ².
        chi2 = np.sum(((flux_obs - flux_pred) / noise_obs) ** 2)
        chi2_grid[i, j] = chi2

In [ ]:
fig, ax = plt.subplots(figsize=(7.0, 5.5))

# Normalize χ² relative to minimum for better visibility.
chi2_min = np.min(chi2_grid)
chi2_norm = chi2_grid - chi2_min

# Heatmap: log scale to show structure across orders of magnitude.
im = ax.contourf(
    log_mstar_grid,
    z_grid,
    chi2_norm,
    levels=np.logspace(-1, 3, 30),
    norm=plt.matplotlib.colors.LogNorm(vmin=0.5, vmax=500),
    cmap="viridis",
)

# Contours at 1σ, 2σ, 3σ (Δχ² = 2.28, 6.18, 11.83 for 2 DoF).
sigma_levels = [2.28, 6.18, 11.83]
cs = ax.contour(
    log_mstar_grid,
    z_grid,
    chi2_norm,
    levels=sigma_levels,
    colors="white",
    linewidths=0.8,
    linestyles=["solid", "dashed", "dotted"],
)

# Manual contour labels (avoid matplotlib's automatic labels).
ax.clabel(cs, inline=True, fontsize=7, fmt=r"$%g\sigma$")

# Mark true (z, M*).
ax.plot(
    log_mstar_true,
    z_true,
    "r*",
    markersize=18,
    markeredgecolor="white",
    markeredgewidth=0.8,
    label=f"True: z={z_true}, M*=10$^{{{log_mstar_true:.1f}}}$ M$_\\odot$",
)

# Colorbar.
cbar = fig.colorbar(im, ax=ax, label=r"$\Delta\chi^2$ (relative to minimum)", pad=0.01)

ax.set_xlabel(r"$\log_{10}(M_* / M_\odot)$")
ax.set_ylabel(r"Redshift $z$")
ax.legend(loc="upper left", fontsize=8, frameon=False)
ax.set_xlim(log_mstar_grid.min(), log_mstar_grid.max())
ax.set_ylim(z_grid.min(), z_grid.max())

plt.savefig("plot_photoz_chi2_grid.png", dpi=150, bbox_inches="tight")